# Gmyrek Task Extraction — 4digits_with_tasks

Dit notebook leest `data/4digits_with_tasks.xlsx` in, splitst de gecombineerde taakkolom (`isco08_5d`) op in:
- `automation_score` — de numerieke automatiseringsscore uit Gmyrek et al. (2023)
- `task_description` — de omschrijving van de taak

Daarnaast bevat het bestand per taak ook de ISCO-08 hiërarchie (1- t/m 4-digit) en twee blootstellingsvariabelen:
- `potential25` — kwalitatieve AI-blootstellingscategorie (Gmyrek)
- `task_color` — geaggregeerd risico-label

Het resultaat wordt opgeslagen als `data/tasks_extracted.xlsx` voor gebruik in vervolganalyses.

In [1]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

## 1. Inlezen

In [2]:
df_raw = pd.read_excel('../data/4digits_with_tasks.xlsx')

print(f'Rijen: {len(df_raw):,}')
print(f'Kolommen: {df_raw.columns.tolist()}')
df_raw.head(3)

Rijen: 3,265
Kolommen: ['isco08_1d', 'isco08_2d', 'isco08_3d', 'isco08_4d', 'isco08_5d', 'potential25', 'task_color']


,isco08_1d,isco08_2d,isco08_3d,isco08_4d,isco08_5d,potential25,task_color
0,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,( 0.34 ) - Presiding over or participating in the proceedings of legislative bodies and administrative councils of n...,Not Exposed,Low
1,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"( 0.3575 ) - Determining, formulating and directing policies of national, state, regional or local governments;",Not Exposed,Low
2,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"( 0.29 ) - Making, ratifying, amending or repealing laws, public rules and regulations within a statutory or constit...",Not Exposed,Low


## 2. Splitsing van `isco08_5d` naar score + taakomschrijving

Het formaat is: `( 0.34 ) - Omschrijving van de taak;`  
We extraheren de score met een regex en halen de taakomschrijving op als de rest na ` - `.

In [3]:
# Patroon: ( score ) - beschrijving
pattern = re.compile(r'^\(\s*([0-9.]+)\s*\)\s*-\s*(.+)$', re.DOTALL)

def parse_task_field(value):
    """Splits '( score ) - beschrijving' in (float score, str beschrijving)."""
    if pd.isna(value):
        return pd.NA, pd.NA
    m = pattern.match(str(value).strip())
    if m:
        score = float(m.group(1))
        desc  = m.group(2).strip().rstrip(';').strip()
        return score, desc
    return pd.NA, str(value).strip()

parsed = df_raw['isco08_5d'].apply(parse_task_field)
df_raw['automation_score'] = [p[0] for p in parsed]
df_raw['task_description'] = [p[1] for p in parsed]

n_failed = df_raw['automation_score'].isna().sum()
print(f'Niet-parseerbare rijen: {n_failed} ({n_failed/len(df_raw)*100:.1f}%)')
df_raw[['isco08_5d', 'automation_score', 'task_description']].head(5)

Niet-parseerbare rijen: 0 (0.0%)


,isco08_5d,automation_score,task_description
0,( 0.34 ) - Presiding over or participating in the proceedings of legislative bodies and administrative councils of n...,0.3400,"Presiding over or participating in the proceedings of legislative bodies and administrative councils of national, st..."
1,"( 0.3575 ) - Determining, formulating and directing policies of national, state, regional or local governments;",0.3575,"Determining, formulating and directing policies of national, state, regional or local governments"
2,"( 0.29 ) - Making, ratifying, amending or repealing laws, public rules and regulations within a statutory or constit...",0.2900,"Making, ratifying, amending or repealing laws, public rules and regulations within a statutory or constitutional fra..."
3,( 0.25 ) - Serving on government administrative boards or official committees;,0.2500,Serving on government administrative boards or official committees
4,( 0.345 ) - Investigating matters of concern to the public and promoting the interests of the constituencies which t...,0.3450,Investigating matters of concern to the public and promoting the interests of the constituencies which they represent


## 3. Opschonen en herordenen van kolommen

In [4]:
df = df_raw[[
    'isco08_1d',
    'isco08_2d',
    'isco08_3d',
    'isco08_4d',
    'task_description',
    'automation_score',
    'potential25',
    'task_color',
]].copy()

df['automation_score'] = pd.to_numeric(df['automation_score'], errors='coerce')

print(df.dtypes)
print()
df.head(5)

isco08_1d            object
isco08_2d            object
isco08_3d            object
isco08_4d            object
task_description     object
automation_score    float64
potential25          object
task_color           object
dtype: object



,isco08_1d,isco08_2d,isco08_3d,isco08_4d,task_description,automation_score,potential25,task_color
0,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Presiding over or participating in the proceedings of legislative bodies and administrative councils of national, st...",0.3400,Not Exposed,Low
1,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Determining, formulating and directing policies of national, state, regional or local governments",0.3575,Not Exposed,Low
2,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Making, ratifying, amending or repealing laws, public rules and regulations within a statutory or constitutional fra...",0.2900,Not Exposed,Low
3,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,Serving on government administrative boards or official committees,0.2500,Not Exposed,Low
4,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,Investigating matters of concern to the public and promoting the interests of the constituencies which they represent,0.3450,Not Exposed,Low


## 4. Basisstatistieken

In [5]:
print('=== automation_score ===')
print(df['automation_score'].describe().round(4))
print()
print('=== potential25 (Gmyrek AI-blootstelling) ===')
print(df['potential25'].value_counts())
print()
print('=== task_color (risico-label) ===')
print(df['task_color'].value_counts())

=== automation_score ===
count    3265.0000
mean        0.2941
std         0.1659
min         0.0475
25%         0.1500
50%         0.2600
75%         0.4050
max         0.7625
Name: automation_score, dtype: float64

=== potential25 (Gmyrek AI-blootstelling) ===
potential25
Not Exposed            1765
Minimal Exposure        707
Exposed: Gradient 2     305
Exposed: Gradient 3     266
Exposed: Gradient 1     143
Exposed: Gradient 4      79
Name: count, dtype: int64

=== task_color (risico-label) ===
task_color
Very Low    1520
Low         1251
Medium       491
High           3
Name: count, dtype: int64


In [6]:
# Gemiddelde automatiseringsscore per 1-digit ISCO beroepsgroep
mean_per_1d = (
    df.groupby('isco08_1d')['automation_score']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'gem_score', 'count': 'n_taken'})
      .round(4)
      .sort_values('gem_score', ascending=False)
)
mean_per_1d

,gem_score,n_taken
isco08_1d,,
4 - Clerical support workers,0.5373,163
2 - Professionals,0.3801,774
1 - Managers,0.3602,293
3 - Technicians and associate professionals,0.3420,586
5 - Service and sales workers,0.2582,273
"8 - Plant and machine operators, and assemblers",0.1953,285
7 - Craft and related trades workers,0.1690,503
"6 - Skilled agricultural, forestry and fishery workers",0.1664,176
9 - Elementary occupations,0.1511,212


## 5. Opslaan

In [7]:
out_path = '../data/tasks_extracted.xlsx'
df.to_excel(out_path, index=False)
print(f'Opgeslagen: {out_path}  ({len(df):,} rijen, {len(df.columns)} kolommen)')

Opgeslagen: ../data/tasks_extracted.xlsx  (3,265 rijen, 8 kolommen)


In [8]:
df

,isco08_1d,isco08_2d,isco08_3d,isco08_4d,task_description,automation_score,potential25,task_color
0,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Presiding over or participating in the proceedings of legislative bodies and administrative councils of national, st...",0.3400,Not Exposed,Low
1,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Determining, formulating and directing policies of national, state, regional or local governments",0.3575,Not Exposed,Low
2,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,"Making, ratifying, amending or repealing laws, public rules and regulations within a statutory or constitutional fra...",0.2900,Not Exposed,Low
3,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,Serving on government administrative boards or official committees,0.2500,Not Exposed,Low
4,1 - Managers,"11 - Chief executives, senior officials and legislators",111 - Legislators and senior officials,1111 - Legislators,Investigating matters of concern to the public and promoting the interests of the constituencies which they represent,0.3450,Not Exposed,Low
...,...,...,...,...,...,...,...,...
3260,9 - Elementary occupations,96 - Refuse workers and other elementary workers,962 - Other elementary workers,9629 - Elementary workers not elsewhere classified,"Directing patrons to restrooms, concession stands and telephones",0.3300,Minimal Exposure,Low
3261,9 - Elementary occupations,96 - Refuse workers and other elementary workers,962 - Other elementary workers,9629 - Elementary workers not elsewhere classified,Directing vehicle drivers to parking spaces,0.2050,Minimal Exposure,Very Low
3262,9 - Elementary occupations,96 - Refuse workers and other elementary workers,962 - Other elementary workers,9629 - Elementary workers not elsewhere classified,Patrolling parking areas in order to prevent vehicle damage and vehicle property thefts,0.1900,Minimal Exposure,Very Low
3263,9 - Elementary occupations,96 - Refuse workers and other elementary workers,962 - Other elementary workers,9629 - Elementary workers not elsewhere classified,Calculating parking charges and collecting fees from customers,0.3750,Minimal Exposure,Low


In [9]:
# Group by isco08_4d and calculate mean automation_score
tasks_by_4d = (
    df.groupby('isco08_4d')['automation_score']
    .mean()
    .reset_index()
    .rename(columns={'automation_score': 'mean_automation_score'})
    .round(4)
)

# make seperate column for isco8-4d code and description and one with both, then sort by mean_automation_score
tasks_by_4d[['isco08_4d_code', 'isco08_4d_desc']] = tasks_by_4d['isco08_4d'].str.split(' - ', n=1, expand=True)
tasks_by_4d['isco08_4d_full'] = tasks_by_4d['isco08_4d_code'] + ' - ' + tasks_by_4d['isco08_4d_desc']
tasks_by_4d = tasks_by_4d[['isco08_4d_code', 'isco08_4d_desc', 'isco08_4d_full', 'mean_automation_score']]
tasks_by_4d = tasks_by_4d.sort_values('mean_automation_score', ascending=False)



# Save to xlsx
tasks_by_4d.to_excel('../data/tasks_by_4digit.xlsx', index=False)
print(f'Opgeslagen: ../data/tasks_by_4digit.xlsx ({len(tasks_by_4d):,} rijen)')
tasks_by_4d.head(10)

tasks_by_4d.describe()

Opgeslagen: ../data/tasks_by_4digit.xlsx (427 rijen)


,mean_automation_score
count,427.000000
mean,0.296584
std,0.144841
min,0.088700
25%,0.175300
50%,0.271500
75%,0.389100
max,0.700000
